# 🔹 Complete Arb Pipeline: 5 Filters + Backtest
### Spread + Cost + Persistence + Z-Score + Vol-Adjusted + P&L

---

**What this filter does:**

At every tick, compute the max spread across all exchange pairs:

$$\text{spread}_{\text{bps}} = \frac{\max(P_A, P_B, P_C) - \min(P_A, P_B, P_C)}{P_{\text{mid}}} \times 10000$$

If $\text{spread} > \text{threshold}$ → **pass**. Otherwise → **discard**.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (16, 5), 'font.size': 11, 'figure.dpi': 100})
print('Imports loaded.')

---
## 1. Generate Data (same as before)

In [ ]:
# ── Config ──
SEED = None
N = 50000
BASE_PRICE = 150.0
DAILY_VOL = 0.015
DT = 1.0 / 390

EXCHANGES = {
    'Alpha': {'bias':  0.00003, 'noise': 0.00025, 'disloc_rate': 0.006, 'disloc_mag': 0.004},
    'Beta':  {'bias': -0.00005, 'noise': 0.00030, 'disloc_rate': 0.008, 'disloc_mag': 0.005},
    'Gamma': {'bias':  0.00001, 'noise': 0.00020, 'disloc_rate': 0.005, 'disloc_mag': 0.003},
}

rng = np.random.default_rng(SEED)

# ── True price (GBM) ──
returns = rng.normal(0, DAILY_VOL * np.sqrt(DT), N)
true_price = BASE_PRICE * np.exp(np.cumsum(returns))

# ── Exchange prices ──
exch_prices = {}
for name, p in EXCHANGES.items():
    bias = p['bias'] * true_price
    noise = rng.normal(0, p['noise'], N) * true_price
    disloc = np.zeros(N)
    n_d = int(N * p['disloc_rate'])
    idxs = rng.choice(N, n_d, replace=False)
    mags = rng.normal(0, p['disloc_mag'], n_d)
    for ix, mg in zip(idxs, mags):
        dl = rng.integers(4, 18)
        end = min(ix + dl, N)
        mr = 0.08 + rng.uniform(0, 0.12)
        decay = np.exp(-np.arange(end - ix) * mr)
        disloc[ix:end] += mg * decay * true_price[ix]
    exch_prices[name] = true_price + bias + noise + disloc

# ── Build DataFrame ──
df = pd.DataFrame({'tick': np.arange(N), 'true_price': true_price})
exch_names = list(EXCHANGES.keys())
for name in exch_names:
    df[name] = exch_prices[name]

print(f'Data: {N:,} ticks, {len(exch_names)} exchanges.')
df.head()

---
## 2. Build Filter 1 — Minimum Spread Threshold

The logic is straightforward:

```
for each tick:
    highest_price = max(Alpha, Beta, Gamma)
    lowest_price  = min(Alpha, Beta, Gamma)
    mid_price     = (highest + lowest) / 2
    spread_bps    = (highest - lowest) / mid * 10000
    
    if spread_bps > THRESHOLD:
        → PASS (potential arb)
    else:
        → DISCARD (no opportunity)
```

In [ ]:
class SpreadFilter:
    """
    Filter 1: Minimum Spread Threshold.

    Computes the max cross-exchange spread at each tick
    and flags ticks where spread > threshold.

    Also identifies which exchange to buy from (cheapest)
    and which to sell on (most expensive).
    """

    def __init__(self, threshold_bps: float = 5.0):
        self.threshold = threshold_bps

    def apply(self, df: pd.DataFrame, exchange_cols: list) -> pd.DataFrame:
        """
        Adds columns to df:
          - max_price, min_price, mid_price
          - spread_bps
          - best_buy  (cheapest exchange)
          - best_sell (most expensive exchange)
          - f1_pass   (True if spread > threshold)

        Returns the modified DataFrame.
        """
        result = df.copy()

        # ── Compute spread ──
        result['max_price'] = result[exchange_cols].max(axis=1)
        result['min_price'] = result[exchange_cols].min(axis=1)
        result['mid_price'] = (result['max_price'] + result['min_price']) / 2
        result['spread_bps'] = (result['max_price'] - result['min_price']) / result['mid_price'] * 10000

        # ── Identify best buy/sell exchanges ──
        result['best_buy'] = result[exchange_cols].idxmin(axis=1)
        result['best_sell'] = result[exchange_cols].idxmax(axis=1)

        # ── Apply threshold ──
        result['f1_pass'] = result['spread_bps'] > self.threshold

        return result

    def summary(self, df: pd.DataFrame):
        """Print filter performance summary."""
        total = len(df)
        passed = df['f1_pass'].sum()
        failed = total - passed

        print('═' * 55)
        print('     FILTER 1 — MINIMUM SPREAD THRESHOLD')
        print('═' * 55)
        print(f'  Threshold           : {self.threshold} bps')
        print(f'  Total ticks         : {total:>10,}')
        print(f'  PASSED              : {passed:>10,} ({passed/total*100:.1f}%)')
        print(f'  DISCARDED           : {failed:>10,} ({failed/total*100:.1f}%)')
        print('─' * 55)

        if passed > 0:
            passed_df = df[df['f1_pass']]
            print(f'  Passed spread stats:')
            print(f'    Mean              : {passed_df["spread_bps"].mean():.2f} bps')
            print(f'    Median            : {passed_df["spread_bps"].median():.2f} bps')
            print(f'    Max               : {passed_df["spread_bps"].max():.1f} bps')
            print(f'    Std               : {passed_df["spread_bps"].std():.2f} bps')
        print('═' * 55)


print('SpreadFilter class defined.')

---
## 3. Apply Filter 1

In [ ]:
THRESHOLD_BPS = 5.0

f1 = SpreadFilter(threshold_bps=THRESHOLD_BPS)
df = f1.apply(df, exch_names)
f1.summary(df)

In [ ]:
# Look at some passed and failed ticks side by side
print('\n PASSED ticks (first 10):')
display(df[df['f1_pass']][['tick','Alpha','Beta','Gamma','spread_bps','best_buy','best_sell']].head(10))

print('\n FAILED ticks (first 10):')
display(df[~df['f1_pass']][['tick','Alpha','Beta','Gamma','spread_bps','best_buy','best_sell']].head(10))

---
## 4. Visualize Filter 1 Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [3, 2, 1.5]})

colors = {'Alpha': '#2196F3', 'Beta': '#F44336', 'Gamma': '#4CAF50'}

# ── Panel 1: Exchange prices with passed ticks highlighted ──
ax = axes[0]
ax.plot(df['tick'], df['true_price'], '--', color='black', alpha=0.3, lw=0.7, label='True Price')
for name in exch_names:
    ax.plot(df['tick'], df[name], color=colors[name], alpha=0.8, lw=0.6, label=name)

# Highlight passed ticks
for _, row in df[df['f1_pass']].iterrows():
    ax.axvspan(row['tick'] - 0.5, row['tick'] + 0.5, alpha=0.08, color='gold')

ax.set_title(f'Exchange Prices — Yellow = Filter 1 PASSED (spread > {THRESHOLD_BPS} bps)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)

# ── Panel 2: Spread with threshold line ──
ax = axes[1]
passed_mask = df['f1_pass'].values
bar_colors = ['#E53935' if p else '#BBDEFB' for p in passed_mask]
ax.bar(df['tick'], df['spread_bps'], width=1.0, color=bar_colors, alpha=0.8)
ax.axhline(THRESHOLD_BPS, color='black', ls='--', lw=1.5,
           label=f'Threshold: {THRESHOLD_BPS} bps')
ax.set_title('Spread (bps) — Red = PASSED, Blue = DISCARDED', fontsize=13, fontweight='bold')
ax.set_ylabel('Spread (bps)')
ax.set_ylim(0, min(df['spread_bps'].quantile(0.995) * 1.2, 80))
ax.legend(fontsize=10)

# ── Panel 3: Binary pass/fail signal ──
ax = axes[2]
ax.fill_between(df['tick'], df['f1_pass'].astype(int), step='mid',
                alpha=0.6, color='#E53935')
ax.set_title('Filter 1 Signal: 1 = PASS, 0 = DISCARD', fontsize=13, fontweight='bold')
ax.set_ylabel('Signal')
ax.set_xlabel('Tick')
ax.set_yticks([0, 1])
ax.set_yticklabels(['DISCARD', 'PASS'])

plt.tight_layout()
plt.show()

---
## 5. Threshold Sensitivity — What if we change the threshold?

In [ ]:
thresholds = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50]
sensitivity = []

for t in thresholds:
    n_pass = (df['spread_bps'] > t).sum()
    passed = df[df['spread_bps'] > t]['spread_bps']
    sensitivity.append({
        'threshold_bps': t,
        'ticks_passed': n_pass,
        'pct_passed': round(n_pass / len(df) * 100, 1),
        'avg_spread': round(passed.mean(), 1) if n_pass > 0 else 0,
        'max_spread': round(passed.max(), 1) if n_pass > 0 else 0,
    })

sens_df = pd.DataFrame(sensitivity)
print('THRESHOLD SENSITIVITY')
print(sens_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ticks passed vs threshold
ax = axes[0]
ax.plot(sens_df['threshold_bps'], sens_df['ticks_passed'], 'o-', color='#2196F3', lw=2, ms=7)
ax.axvline(THRESHOLD_BPS, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our threshold: {THRESHOLD_BPS} bps')
ax.set_xlabel('Threshold (bps)')
ax.set_ylabel('Ticks Passed')
ax.set_title('Filter Strictness: Ticks Passed vs Threshold', fontweight='bold')
ax.legend(fontsize=9)

# Avg spread of passed ticks vs threshold
ax = axes[1]
ax.plot(sens_df['threshold_bps'], sens_df['avg_spread'], 's-', color='#4CAF50', lw=2, ms=7)
ax.axvline(THRESHOLD_BPS, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our threshold: {THRESHOLD_BPS} bps')
ax.set_xlabel('Threshold (bps)')
ax.set_ylabel('Avg Spread of Passed Ticks (bps)')
ax.set_title('Signal Quality: Avg Spread vs Threshold', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(' Higher threshold = fewer signals but higher quality.')
print(' Lower threshold = more signals but more noise.')

---
## 6. Arb Pair Breakdown — Which exchanges are diverging?

In [ ]:
passed_df = df[df['f1_pass']].copy()

if not passed_df.empty:
    # Count by pair
    pair_counts = passed_df.groupby(['best_buy', 'best_sell']).agg(
        count=('spread_bps', 'count'),
        avg_spread=('spread_bps', 'mean'),
        max_spread=('spread_bps', 'max')
    ).round(1).sort_values('count', ascending=False)

    print('FILTER 1 PASSED — BY EXCHANGE PAIR')
    print('  (Buy cheap at → Sell expensive at)\n')
    print(pair_counts.to_string())

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [f'{b}→{s}' for (b, s) in pair_counts.index]
    bar_colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
    ax.barh(labels, pair_counts['count'], color=bar_colors[:len(labels)], edgecolor='white')
    for i, (cnt, avg) in enumerate(zip(pair_counts['count'], pair_counts['avg_spread'])):
        ax.text(cnt + 2, i, f'{cnt} ticks (avg {avg:.0f} bps)', va='center', fontsize=10)
    ax.set_title('Arb Opportunities by Direction (Filter 1 Passed)', fontweight='bold')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

---
## 7. Export Filtered Data for Next Stage

In [ ]:
# Save both full data (with filter column) and passed-only data
export_cols = ['tick', 'true_price', 'Alpha', 'Beta', 'Gamma',
               'spread_bps', 'best_buy', 'best_sell', 'f1_pass']

df[export_cols].to_csv('filter1_full.csv', index=False)
df[df['f1_pass']][export_cols].to_csv('filter1_passed.csv', index=False)

n_passed = df['f1_pass'].sum()
print(f'Saved filter1_full.csv    — {len(df):,} rows (all ticks with f1_pass column)')
print(f'Saved filter1_passed.csv  — {n_passed:,} rows (passed ticks only)')
print(f'\n   → These go into Filter 2 (Cost Adjustment) next.')

---
---

# 🔹 Filter 2 — Cost Adjustment
### Arbitrage Signal Pipeline: Step 2 of 5

---

**What this filter does:**

A gross spread of 6 bps is worthless if it costs 7 bps to execute. Filter 2 subtracts real trading costs:

$$\text{net\_spread} = \text{gross\_spread} - \underbrace{(\text{fee}_{\text{buy}} + \text{fee}_{\text{sell}})}_{\text{exchange fees}} - \underbrace{2 \times \text{slippage}}_{\text{both legs}}$$

If $\text{net\_spread} > 0$ → **pass**. Otherwise → **discard** (the arb doesn't survive costs).

---

---
## 8. Define Exchange Fee Structure

In [ ]:
# ── Exchange Fees (taker fees in bps) ──
# These are what you pay per leg when you cross the spread
EXCHANGE_FEES_BPS = {
    'Alpha': 2.0,    # 2 bps per side
    'Beta':  3.0,    # 3 bps per side (more expensive venue)
    'Gamma': 1.5,    # 1.5 bps per side (cheapest venue)
}

# ── Slippage ──
# Market impact when your order executes
SLIPPAGE_BPS = 1.0   # 1 bps per leg

print('COST STRUCTURE')
print('─' * 40)
for exch, fee in EXCHANGE_FEES_BPS.items():
    print(f'  {exch:<10s}  fee = {fee:.1f} bps/side')
print(f'  {"Slippage":<10s}  = {SLIPPAGE_BPS:.1f} bps/side')
print('─' * 40)
print(f'\n  Example round-trip cost (Alpha → Beta):')
cost_ab = EXCHANGE_FEES_BPS['Alpha'] + EXCHANGE_FEES_BPS['Beta'] + 2 * SLIPPAGE_BPS
print(f'    = fee_buy({EXCHANGE_FEES_BPS["Alpha"]}) + fee_sell({EXCHANGE_FEES_BPS["Beta"]}) + 2×slippage({SLIPPAGE_BPS}) = {cost_ab:.1f} bps')

---
## 9. Build Filter 2 — Cost Adjustment

In [ ]:
class CostFilter:
    """
    Filter 2: Cost Adjustment.

    For each tick that passed Filter 1:
      1. Look up the buy exchange (cheapest) and sell exchange (most expensive)
      2. Compute total round-trip cost:
           cost = fee_buy + fee_sell + 2 × slippage
      3. Compute net spread:
           net_spread = gross_spread - cost
      4. Pass only if net_spread > 0
    """

    def __init__(self, fees_bps: dict, slippage_bps: float):
        self.fees = fees_bps
        self.slippage = slippage_bps

    def compute_cost(self, buy_exch: str, sell_exch: str) -> float:
        """Total round-trip cost in bps for a given pair."""
        return self.fees[buy_exch] + self.fees[sell_exch] + 2 * self.slippage

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Adds columns:
          - cost_bps       : total round-trip cost for the best pair
          - net_spread_bps : gross spread minus cost
          - f2_pass        : True if net_spread > 0 AND f1_pass is True
        """
        result = df.copy()

        # Compute cost per tick based on which exchanges are involved
        result['cost_bps'] = result.apply(
            lambda row: self.compute_cost(row['best_buy'], row['best_sell']),
            axis=1
        )

        # Net spread = gross - cost
        result['net_spread_bps'] = result['spread_bps'] - result['cost_bps']

        # Pass only if: passed Filter 1 AND net spread is positive
        result['f2_pass'] = result['f1_pass'] & (result['net_spread_bps'] > 0)

        return result

    def summary(self, df: pd.DataFrame):
        """Print filter performance summary."""
        f1_passed = df['f1_pass'].sum()
        f2_passed = df['f2_pass'].sum()
        killed = f1_passed - f2_passed

        print('═' * 55)
        print('     FILTER 2 — COST ADJUSTMENT')
        print('═' * 55)
        print(f'  Input (from F1)     : {f1_passed:>10,} ticks')
        print(f'  PASSED (net > 0)    : {f2_passed:>10,} ({f2_passed/f1_passed*100:.1f}% of F1)' if f1_passed > 0 else '  PASSED: 0')
        print(f'  KILLED by costs     : {killed:>10,} ({killed/f1_passed*100:.1f}% of F1)' if f1_passed > 0 else '  KILLED: 0')
        print('─' * 55)

        if f2_passed > 0:
            p = df[df['f2_pass']]
            print(f'  Passed tick stats:')
            print(f'    Gross spread  : mean={p["spread_bps"].mean():.1f}, median={p["spread_bps"].median():.1f} bps')
            print(f'    Cost          : mean={p["cost_bps"].mean():.1f} bps')
            print(f'    Net spread    : mean={p["net_spread_bps"].mean():.1f}, median={p["net_spread_bps"].median():.1f} bps')
            print(f'    Max net spread: {p["net_spread_bps"].max():.1f} bps')

        # Show what got killed
        if killed > 0:
            k = df[df['f1_pass'] & ~df['f2_pass']]
            print(f'\n  Killed tick stats (arb didn\'t survive costs):')
            print(f'    Gross spread  : mean={k["spread_bps"].mean():.1f} bps')
            print(f'    Cost          : mean={k["cost_bps"].mean():.1f} bps')
            print(f'    Net spread    : mean={k["net_spread_bps"].mean():.1f} bps (negative = loss)')

        print('═' * 55)


print('CostFilter class defined.')

---
## 10. Apply Filter 2

In [ ]:
f2 = CostFilter(fees_bps=EXCHANGE_FEES_BPS, slippage_bps=SLIPPAGE_BPS)
df = f2.apply(df)
f2.summary(df)

In [ ]:
# Show ticks that were KILLED by costs (passed F1 but failed F2)
killed = df[df['f1_pass'] & ~df['f2_pass']]
print(f'\n Ticks KILLED by costs ({len(killed)} ticks):')
print('   These had a visible spread but it didn\'t survive transaction costs.\n')
display(killed[['tick','Alpha','Beta','Gamma','spread_bps','cost_bps','net_spread_bps','best_buy','best_sell']].head(15))

In [ ]:
# Show ticks that SURVIVED costs
survived = df[df['f2_pass']]
print(f'\n Ticks that SURVIVED costs ({len(survived)} ticks):\n')
display(survived[['tick','Alpha','Beta','Gamma','spread_bps','cost_bps','net_spread_bps','best_buy','best_sell']].head(15))

---
## 11. Visualize Filter 2 Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [2.5, 2.5, 1.5]})

# ── Panel 1: Gross vs Net spread ──
ax = axes[0]
f1_data = df[df['f1_pass']].copy()

if not f1_data.empty:
    bar_colors = ['#4CAF50' if p else '#F44336' for p in f1_data['f2_pass']]
    ax.bar(f1_data['tick'], f1_data['spread_bps'], width=1.2, alpha=0.4,
           color='#90CAF9', label='Gross Spread')
    ax.bar(f1_data['tick'], f1_data['net_spread_bps'], width=1.2, alpha=0.8,
           color=bar_colors, label='Net Spread (green=pass, red=killed)')
    ax.axhline(0, color='black', lw=1)

ax.set_title('Gross Spread vs Net Spread After Costs (Filter 1 ticks only)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Spread (bps)')
ax.legend(fontsize=9)

# ── Panel 2: Cost breakdown by exchange pair ──
ax = axes[1]
if not f1_data.empty:
    f1_data['pair'] = f1_data['best_buy'] + '→' + f1_data['best_sell']
    pair_groups = f1_data.groupby('pair').agg(
        avg_gross=('spread_bps', 'mean'),
        avg_cost=('cost_bps', 'mean'),
        avg_net=('net_spread_bps', 'mean'),
        count=('tick', 'count')
    ).sort_values('count', ascending=True)

    y_pos = range(len(pair_groups))
    ax.barh(y_pos, pair_groups['avg_gross'], height=0.6, alpha=0.4,
            color='#90CAF9', label='Avg Gross Spread')
    net_colors = ['#4CAF50' if v > 0 else '#F44336' for v in pair_groups['avg_net']]
    ax.barh(y_pos, pair_groups['avg_net'], height=0.6, alpha=0.85,
            color=net_colors, label='Avg Net Spread')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'{p} (n={c})' for p, c in zip(pair_groups.index, pair_groups['count'])], fontsize=9)
    ax.axvline(0, color='black', lw=0.8)
    # Add cost annotation
    for i, (_, row) in enumerate(pair_groups.iterrows()):
        ax.text(row['avg_gross'] + 0.3, i,
                f'cost: {row["avg_cost"]:.1f} bps',
                va='center', fontsize=8, color='gray')

ax.set_title('Avg Gross vs Net Spread by Exchange Pair', fontsize=13, fontweight='bold')
ax.set_xlabel('Spread (bps)')
ax.legend(fontsize=9, loc='lower right')

# ── Panel 3: Filter 1 vs Filter 2 signal ──
ax = axes[2]
ax.fill_between(df['tick'], df['f1_pass'].astype(int) * 1.0, step='mid',
                alpha=0.3, color='#FF9800', label='F1 Pass')
ax.fill_between(df['tick'], df['f2_pass'].astype(int) * 0.9, step='mid',
                alpha=0.6, color='#4CAF50', label='F2 Pass (survives costs)')
ax.set_title('Filter 1 vs Filter 2 Signal', fontsize=13, fontweight='bold')
ax.set_ylabel('Signal')
ax.set_xlabel('Tick')
ax.set_yticks([0, 1])
ax.set_yticklabels(['DISCARD', 'PASS'])
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Net spread distribution: passed vs killed ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

f1_only = df[df['f1_pass']]

ax = axes[0]
ax.hist(f1_only['net_spread_bps'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', ls='--', lw=1.5, label='Break-even (net = 0)')
ax.set_title('Net Spread Distribution (all F1 ticks)', fontweight='bold')
ax.set_xlabel('Net Spread (bps)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

ax = axes[1]
if df['f2_pass'].sum() > 0:
    ax.hist(df[df['f2_pass']]['net_spread_bps'], bins=40, color='#4CAF50',
            edgecolor='white', alpha=0.85, label='F2 Passed')
if (df['f1_pass'] & ~df['f2_pass']).sum() > 0:
    ax.hist(df[df['f1_pass'] & ~df['f2_pass']]['net_spread_bps'], bins=20,
            color='#F44336', edgecolor='white', alpha=0.65, label='Killed by costs')
ax.axvline(0, color='black', ls='--', lw=1)
ax.set_title('Passed vs Killed by Costs', fontweight='bold')
ax.set_xlabel('Net Spread (bps)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 12. Combined Funnel — Filter 1 + Filter 2

In [ ]:
total = len(df)
f1_p = df['f1_pass'].sum()
f2_p = df['f2_pass'].sum()
f1_kill = total - f1_p
f2_kill = f1_p - f2_p

print('SIGNAL FUNNEL')
print('═' * 50)
print(f'  All ticks             : {total:>8,}')
print(f'  ↓ Filter 1 removed    : {f1_kill:>8,} ({f1_kill/total*100:.1f}%)')
print(f'  After Filter 1        : {f1_p:>8,} ({f1_p/total*100:.1f}%)')
print(f'  ↓ Filter 2 removed    : {f2_kill:>8,} ({f2_kill/f1_p*100:.1f}% of F1)' if f1_p > 0 else '')
print(f'  After Filter 1+2      : {f2_p:>8,} ({f2_p/total*100:.1f}% of total)')
print('═' * 50)

# Funnel bar chart
fig, ax = plt.subplots(figsize=(8, 5))
stages = ['All Ticks', 'After F1\n(spread > threshold)', 'After F2\n(net > 0 after costs)']
counts = [total, f1_p, f2_p]
bar_colors = ['#90CAF9', '#FF9800', '#4CAF50']

bars = ax.bar(stages, counts, color=bar_colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}', ha='center', fontsize=12, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            f'{count/total*100:.1f}%', ha='center', fontsize=11, color='white', fontweight='bold')

ax.set_title('Signal Funnel: Filter 1 → Filter 2', fontsize=13, fontweight='bold')
ax.set_ylabel('Ticks Remaining')
ax.set_ylim(0, total * 1.15)
plt.tight_layout()
plt.show()

---
## 13. Export Updated Data

In [ ]:
export_cols = ['tick', 'true_price', 'Alpha', 'Beta', 'Gamma',
               'spread_bps', 'cost_bps', 'net_spread_bps',
               'best_buy', 'best_sell', 'f1_pass', 'f2_pass']

df[export_cols].to_csv('filter1_2_full.csv', index=False)
df[df['f2_pass']][export_cols].to_csv('filter1_2_passed.csv', index=False)

print(f'Saved filter1_2_full.csv   — {len(df):,} rows (all ticks, both filter columns)')
print(f'Saved filter1_2_passed.csv — {f2_p:,} rows (passed both filters)')
print(f'\n   → These go into Filter 3 (Persistence Check) next.')

---
---

# 🔹 Filter 3 — Persistence Check
### Arbitrage Signal Pipeline: Step 3 of 5

---

**What this filter does:**

A spread that flickers above threshold for 1 tick then vanishes is almost certainly noise: a stale quote, a momentary order book imbalance. Real dislocations persist.

$$\text{Pass if the spread has been above threshold for } N \text{ consecutive ticks}$$

This is a **lookback** filter, not a point-in-time check. It requires the opportunity to prove itself by surviving for multiple ticks before we act.

---

---
## 14. Build Filter 3 — Persistence Check

In [ ]:
class PersistenceFilter:
    """
    Filter 3: Persistence Check.

    For each tick, count how many consecutive ticks (including this one)
    the spread has been above the Filter 1 threshold AND survived costs (F2).

    Only pass if the streak >= required_ticks.

    Why this matters:
      - 1-tick spikes are microstructure noise (stale quotes, phantom liquidity)
      - Real dislocations persist for multiple ticks as the market absorbs the shock
      - By waiting, we confirm the opportunity is real before risking capital

    Tradeoff:
      - Higher persistence = fewer false signals, but we enter later (smaller remaining spread)
      - Lower persistence = more signals, but more noise
    """

    def __init__(self, required_ticks: int = 3):
        self.required = required_ticks

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Adds columns:
          - streak        : consecutive ticks where f2_pass has been True
          - f3_pass       : True if streak >= required AND f2_pass is True
          - streak_start  : tick where current streak began
        """
        result = df.copy()

        f2 = result['f2_pass'].values
        n = len(f2)

        streak = np.zeros(n, dtype=int)
        streak_start = np.full(n, -1, dtype=int)

        # ── Count consecutive runs ──
        current_streak = 0
        current_start = 0

        for i in range(n):
            if f2[i]:
                if current_streak == 0:
                    current_start = i
                current_streak += 1
            else:
                current_streak = 0

            streak[i] = current_streak
            if current_streak > 0:
                streak_start[i] = current_start

        result['streak'] = streak
        result['streak_start'] = streak_start
        result['f3_pass'] = result['f2_pass'] & (streak >= self.required)

        return result

    def summary(self, df: pd.DataFrame):
        """Print filter performance summary."""
        f2_p = df['f2_pass'].sum()
        f3_p = df['f3_pass'].sum()
        killed = f2_p - f3_p

        print('═' * 55)
        print('     FILTER 3 — PERSISTENCE CHECK')
        print('═' * 55)
        print(f'  Required streak     : {self.required} consecutive ticks')
        print(f'  Input (from F2)     : {f2_p:>10,} ticks')
        print(f'  PASSED (persistent) : {f3_p:>10,} ({f3_p/f2_p*100:.1f}% of F2)' if f2_p > 0 else '  PASSED: 0')
        print(f'  KILLED (too short)  : {killed:>10,} ({killed/f2_p*100:.1f}% of F2)' if f2_p > 0 else '  KILLED: 0')
        print('─' * 55)

        if f3_p > 0:
            p = df[df['f3_pass']]
            print(f'  Passed tick stats:')
            print(f'    Net spread    : mean={p["net_spread_bps"].mean():.1f}, median={p["net_spread_bps"].median():.1f} bps')
            print(f'    Avg streak    : {p["streak"].mean():.1f} ticks')
            print(f'    Max streak    : {p["streak"].max()} ticks')

        # Analyze what got killed
        killed_df = df[df['f2_pass'] & ~df['f3_pass']]
        if len(killed_df) > 0:
            print(f'\n  Killed tick stats (too short-lived):')
            print(f'    Avg streak    : {killed_df["streak"].mean():.1f} ticks')
            print(f'    Net spread    : mean={killed_df["net_spread_bps"].mean():.1f} bps')
            print(f'    → These were likely noise / stale quotes')

        print('═' * 55)


print('PersistenceFilter class defined.')

---
## 15. Apply Filter 3

In [ ]:
PERSISTENCE_TICKS = 3

f3 = PersistenceFilter(required_ticks=PERSISTENCE_TICKS)
df = f3.apply(df)
f3.summary(df)

In [ ]:
# ── Show examples of killed vs passed ──

# Ticks killed: had a profitable spread but it only lasted 1-2 ticks
killed = df[df['f2_pass'] & ~df['f3_pass']]
print(f'KILLED by persistence ({len(killed)} ticks) — spread was real but too brief:\n')
display(killed[['tick','Alpha','Beta','Gamma','net_spread_bps','streak','best_buy','best_sell']].head(15))

print(f'\n PASSED persistence ({df["f3_pass"].sum()} ticks) — spread held for {PERSISTENCE_TICKS}+ ticks:\n')
passed = df[df['f3_pass']]
display(passed[['tick','Alpha','Beta','Gamma','net_spread_bps','streak','streak_start','best_buy','best_sell']].head(15))

---
## 16. Visualize Filter 3 Results

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(18, 18), gridspec_kw={'height_ratios': [3, 2, 1.5, 1.5]})

colors_exch = {'Alpha': '#2196F3', 'Beta': '#F44336', 'Gamma': '#4CAF50'}

# ── Panel 1: Prices with F3 pass highlighted ──
ax = axes[0]
ax.plot(df['tick'], df['true_price'], '--', color='black', alpha=0.3, lw=0.7, label='True Price')
for name in exch_names:
    ax.plot(df['tick'], df[name], color=colors_exch[name], alpha=0.8, lw=0.6, label=name)
for _, row in df[df['f3_pass']].iterrows():
    ax.axvspan(row['tick'] - 0.5, row['tick'] + 0.5, alpha=0.12, color='#4CAF50')
ax.set_title(f'Exchange Prices — Green = Filter 3 PASSED (persistent for {PERSISTENCE_TICKS}+ ticks)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)

# ── Panel 2: Streak lengths over time ──
ax = axes[1]
streak_colors = ['#4CAF50' if p else '#FFCDD2' for p in df['f3_pass']]
f2_ticks = df[df['f2_pass']]
ax.bar(f2_ticks['tick'], f2_ticks['streak'], width=1.0, color=[
    '#4CAF50' if p else '#FFCDD2' for p in f2_ticks['f3_pass']], alpha=0.8)
ax.axhline(PERSISTENCE_TICKS, color='black', ls='--', lw=1.5,
           label=f'Required: {PERSISTENCE_TICKS} ticks')
ax.set_title('Streak Length (consecutive F2-pass ticks)', fontsize=13, fontweight='bold')
ax.set_ylabel('Streak (ticks)')
ax.legend(fontsize=10)

# ── Panel 3: F1 vs F2 vs F3 signals ──
ax = axes[2]
ax.fill_between(df['tick'], df['f1_pass'].astype(float) * 1.0, step='mid',
                alpha=0.2, color='#FF9800', label='F1 Pass')
ax.fill_between(df['tick'], df['f2_pass'].astype(float) * 0.95, step='mid',
                alpha=0.3, color='#2196F3', label='F2 Pass')
ax.fill_between(df['tick'], df['f3_pass'].astype(float) * 0.9, step='mid',
                alpha=0.6, color='#4CAF50', label='F3 Pass')
ax.set_title('Progressive Filtering: F1 → F2 → F3', fontsize=13, fontweight='bold')
ax.set_ylabel('Signal')
ax.set_yticks([0, 1])
ax.set_yticklabels(['DISCARD', 'PASS'])
ax.legend(fontsize=9, loc='upper right')

# ── Panel 4: Zoomed example of persistence in action ──
ax = axes[3]
# Find a good window with both passed and killed streaks
f2_ticks_list = df[df['f2_pass']]['tick'].values
if len(f2_ticks_list) > 20:
    center = f2_ticks_list[len(f2_ticks_list)//3]  # pick a spot 1/3 through
    zs, ze = max(0, center - 40), min(len(df), center + 60)
else:
    zs, ze = 0, min(200, len(df))

zdf = df.iloc[zs:ze]
for name in exch_names:
    ax.plot(zdf['tick'], zdf[name], color=colors_exch[name], lw=1.2, alpha=0.9, label=name)

for _, row in zdf[zdf['f2_pass'] & ~zdf['f3_pass']].iterrows():
    ax.axvspan(row['tick']-0.5, row['tick']+0.5, alpha=0.2, color='#F44336')
for _, row in zdf[zdf['f3_pass']].iterrows():
    ax.axvspan(row['tick']-0.5, row['tick']+0.5, alpha=0.2, color='#4CAF50')

ax.set_title(f'Zoomed: Red = killed (streak < {PERSISTENCE_TICKS}), Green = passed (streak ≥ {PERSISTENCE_TICKS})',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.set_xlabel('Tick')
ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

---
## 17. Persistence Sensitivity — What's the right streak length?

In [ ]:
streak_values = [1, 2, 3, 4, 5, 7, 10]
persist_sens = []

for sv in streak_values:
    pf = PersistenceFilter(required_ticks=sv)
    test_df = pf.apply(df)
    n_pass = test_df['f3_pass'].sum()
    if n_pass > 0:
        p = test_df[test_df['f3_pass']]
        persist_sens.append({
            'required_streak': sv,
            'ticks_passed': n_pass,
            'pct_of_f2': round(n_pass / df['f2_pass'].sum() * 100, 1) if df['f2_pass'].sum() > 0 else 0,
            'avg_net_spread': round(p['net_spread_bps'].mean(), 1),
            'avg_streak': round(p['streak'].mean(), 1),
        })
    else:
        persist_sens.append({
            'required_streak': sv,
            'ticks_passed': 0,
            'pct_of_f2': 0,
            'avg_net_spread': 0,
            'avg_streak': 0,
        })

ps_df = pd.DataFrame(persist_sens)
print('PERSISTENCE SENSITIVITY')
print(ps_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(ps_df['required_streak'], ps_df['ticks_passed'], 'o-', color='#2196F3', lw=2, ms=7)
ax.axvline(PERSISTENCE_TICKS, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: {PERSISTENCE_TICKS} ticks')
ax.set_xlabel('Required Consecutive Ticks')
ax.set_ylabel('Ticks Passed')
ax.set_title('Signals vs Required Persistence', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(ps_df['required_streak'], ps_df['avg_net_spread'], 's-', color='#4CAF50', lw=2, ms=7)
ax.axvline(PERSISTENCE_TICKS, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: {PERSISTENCE_TICKS} ticks')
ax.set_xlabel('Required Consecutive Ticks')
ax.set_ylabel('Avg Net Spread of Passed Ticks (bps)')
ax.set_title('Signal Quality vs Required Persistence', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Longer persistence = fewer but higher-quality signals.')
print('The sweet spot is where quality rises without losing too many signals.')

---
## 18. Combined Funnel — Filters 1 + 2 + 3

In [ ]:
total = len(df)
f1_p = df['f1_pass'].sum()
f2_p = df['f2_pass'].sum()
f3_p = df['f3_pass'].sum()

print('SIGNAL FUNNEL')
print('═' * 55)
print(f'  All ticks             : {total:>8,}')
print(f'  ↓ Filter 1 removed    : {total - f1_p:>8,}')
print(f'  After Filter 1        : {f1_p:>8,} ({f1_p/total*100:.1f}%)')
print(f'  ↓ Filter 2 removed    : {f1_p - f2_p:>8,}')
print(f'  After Filter 1+2      : {f2_p:>8,} ({f2_p/total*100:.1f}%)')
print(f'  ↓ Filter 3 removed    : {f2_p - f3_p:>8,}')
print(f'  After Filter 1+2+3    : {f3_p:>8,} ({f3_p/total*100:.1f}%)')
print('═' * 55)

# Funnel chart
fig, ax = plt.subplots(figsize=(10, 5))
stages = ['All Ticks', 'After F1\n(spread > thresh)', 'After F2\n(net > 0)', 'After F3\n(persists 3+ ticks)']
counts = [total, f1_p, f2_p, f3_p]
bar_colors = ['#90CAF9', '#FF9800', '#2196F3', '#4CAF50']

bars = ax.bar(stages, counts, color=bar_colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}', ha='center', fontsize=12, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            f'{count/total*100:.1f}%', ha='center', fontsize=11, color='white', fontweight='bold')

ax.set_title('Signal Funnel: Filter 1 → 2 → 3', fontsize=13, fontweight='bold')
ax.set_ylabel('Ticks Remaining')
ax.set_ylim(0, total * 1.15)
plt.tight_layout()
plt.show()

---
## 19. Export Updated Data

In [ ]:
export_cols = ['tick', 'true_price', 'Alpha', 'Beta', 'Gamma',
               'spread_bps', 'cost_bps', 'net_spread_bps',
               'best_buy', 'best_sell', 'streak',
               'f1_pass', 'f2_pass', 'f3_pass']

df[export_cols].to_csv('filter1_2_3_full.csv', index=False)
df[df['f3_pass']][export_cols].to_csv('filter1_2_3_passed.csv', index=False)

print(f'Saved filter1_2_3_full.csv   — {len(df):,} rows (all ticks, all filter columns)')
print(f'Saved filter1_2_3_passed.csv — {f3_p:,} rows (passed all 3 filters)')
print(f'\n   → These go into Filter 4 (Z-Score) next.')

---
---

# 🔹 Filter 4 — Z-Score Filter
### Arbitrage Signal Pipeline: Step 4 of 5

---

**What this filter does:**

A 7 bps spread means nothing if the spread between those exchanges *normally* bounces around 0–10 bps. The z-score normalizes the spread relative to its own recent history:

$$z_t = \frac{\text{spread}_t - \mu_{\text{rolling}}}{\sigma_{\text{rolling}}}$$

where $\mu$ and $\sigma$ are computed over a rolling lookback window.

Pass only if $|z| > \text{threshold}$ (e.g. 2.0), meaning the current spread is more than 2 standard deviations from its recent average — **statistically unusual**, not just business as usual.

---

---
## 20. Build Filter 4 — Z-Score

In [ ]:
from collections import deque

class ZScoreFilter:
    """
    Filter 4: Z-Score Filter.

    For each exchange pair, maintains a rolling window of recent spreads.
    Computes z-score = (current_spread - rolling_mean) / rolling_std.

    Only passes if:
      - f3_pass is True (survived all prior filters)
      - |z-score| > threshold (spread is statistically unusual)

    Why this matters:
      Exchange Alpha-Beta might always have a 4-6 bps spread due to
      structural differences (different fees, different client base).
      A 7 bps spread on that pair is NORMAL, not an arb.
      But a 7 bps spread on Alpha-Gamma, which usually has 1-2 bps,
      IS unusual — z-score will be high.

    The z-score answers: 'Is this spread unusual for THIS pair?'
    """

    def __init__(self, lookback: int = 120, z_threshold: float = 2.0):
        self.lookback = lookback
        self.z_threshold = z_threshold

    def apply(self, df: pd.DataFrame, exchange_cols: list) -> pd.DataFrame:
        """
        Computes per-pair z-scores and a composite z-score for the
        best buy→sell pair at each tick.

        Adds columns:
          - rolling_mean    : rolling mean of the max spread
          - rolling_std     : rolling std of the max spread
          - zscore          : (spread - mean) / std
          - zscore_abs      : |zscore|
          - f4_pass         : True if f3_pass AND |zscore| > threshold
        """
        result = df.copy()

        # ── Compute rolling stats on the spread series ──
        spread = result['spread_bps'].values
        n = len(spread)

        rolling_mean = np.full(n, np.nan)
        rolling_std = np.full(n, np.nan)
        zscore = np.full(n, 0.0)

        # Use expanding window until we have enough data, then rolling
        window = deque(maxlen=self.lookback)

        for i in range(n):
            window.append(spread[i])

            if len(window) >= 30:  # minimum data for meaningful stats
                arr = np.array(window)
                mu = arr.mean()
                sigma = arr.std()
                rolling_mean[i] = mu
                rolling_std[i] = sigma

                if sigma > 1e-8:
                    zscore[i] = (spread[i] - mu) / sigma

        result['rolling_mean'] = rolling_mean
        result['rolling_std'] = rolling_std
        result['zscore'] = zscore
        result['zscore_abs'] = np.abs(zscore)

        # ── Apply filter ──
        result['f4_pass'] = result['f3_pass'] & (result['zscore_abs'] > self.z_threshold)

        return result

    def summary(self, df: pd.DataFrame):
        """Print filter performance summary."""
        f3_p = df['f3_pass'].sum()
        f4_p = df['f4_pass'].sum()
        killed = f3_p - f4_p

        print('═' * 55)
        print('     FILTER 4 — Z-SCORE FILTER')
        print('═' * 55)
        print(f'  Lookback window     : {self.lookback} ticks')
        print(f'  Z-score threshold   : |z| > {self.z_threshold}')
        print(f'  Input (from F3)     : {f3_p:>10,} ticks')
        print(f'  PASSED (unusual)    : {f4_p:>10,} ({f4_p/f3_p*100:.1f}% of F3)' if f3_p > 0 else '  PASSED: 0')
        print(f'  KILLED (normal)     : {killed:>10,} ({killed/f3_p*100:.1f}% of F3)' if f3_p > 0 else '  KILLED: 0')
        print('─' * 55)

        if f4_p > 0:
            p = df[df['f4_pass']]
            print(f'  Passed tick stats:')
            print(f'    Net spread    : mean={p["net_spread_bps"].mean():.1f}, max={p["net_spread_bps"].max():.1f} bps')
            print(f'    |z-score|     : mean={p["zscore_abs"].mean():.2f}, max={p["zscore_abs"].max():.2f}')
            print(f'    Streak        : mean={p["streak"].mean():.1f} ticks')

        if killed > 0:
            k = df[df['f3_pass'] & ~df['f4_pass']]
            print(f'\n  Killed tick stats (spread was normal for this pair):')
            print(f'    Net spread    : mean={k["net_spread_bps"].mean():.1f} bps')
            print(f'    |z-score|     : mean={k["zscore_abs"].mean():.2f}')
            print(f'    → These spreads were within normal range')

        print('═' * 55)


print('ZScoreFilter class defined.')

---
## 21. Apply Filter 4

In [ ]:
ZSCORE_LOOKBACK = 120
ZSCORE_THRESHOLD = 2.0

f4 = ZScoreFilter(lookback=ZSCORE_LOOKBACK, z_threshold=ZSCORE_THRESHOLD)
df = f4.apply(df, exch_names)
f4.summary(df)

In [ ]:
# ── Show what got killed vs passed ──
killed_f4 = df[df['f3_pass'] & ~df['f4_pass']]
passed_f4 = df[df['f4_pass']]

print(f'KILLED by z-score ({len(killed_f4)} ticks) — spread was within normal range:\n')
if len(killed_f4) > 0:
    display(killed_f4[['tick','spread_bps','net_spread_bps','zscore','rolling_mean','rolling_std',
                        'best_buy','best_sell']].head(15))

print(f'\n PASSED z-score ({len(passed_f4)} ticks) — spread is statistically unusual:\n')
if len(passed_f4) > 0:
    display(passed_f4[['tick','spread_bps','net_spread_bps','zscore','rolling_mean','rolling_std',
                        'streak','best_buy','best_sell']].head(15))

---
## 22. Visualize Filter 4 Results

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(18, 20), gridspec_kw={'height_ratios': [2.5, 2.5, 2, 1.5]})

colors_exch = {'Alpha': '#2196F3', 'Beta': '#F44336', 'Gamma': '#4CAF50'}

# ── Panel 1: Spread with rolling mean ± std bands ──
ax = axes[0]
ax.plot(df['tick'], df['spread_bps'], color='#90CAF9', alpha=0.5, lw=0.5, label='Raw Spread')
ax.plot(df['tick'], df['rolling_mean'], color='#1565C0', lw=1.2, label=f'Rolling Mean ({ZSCORE_LOOKBACK} ticks)')

# ± 2 std bands
upper = df['rolling_mean'] + ZSCORE_THRESHOLD * df['rolling_std']
lower = (df['rolling_mean'] - ZSCORE_THRESHOLD * df['rolling_std']).clip(lower=0)
ax.fill_between(df['tick'], lower, upper, alpha=0.12, color='#1565C0', label=f'±{ZSCORE_THRESHOLD}σ band')

# Mark F4 passes
for _, row in df[df['f4_pass']].iterrows():
    ax.plot(row['tick'], row['spread_bps'], 'o', color='#4CAF50', ms=4, alpha=0.7)

ax.set_title(f'Spread with Rolling Mean ± {ZSCORE_THRESHOLD}σ — Green dots = F4 PASSED',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Spread (bps)')
ax.set_ylim(0, min(df['spread_bps'].quantile(0.995) * 1.3, 60))
ax.legend(fontsize=9)

# ── Panel 2: Z-score time series ──
ax = axes[1]
ax.plot(df['tick'], df['zscore'], color='#607D8B', alpha=0.6, lw=0.5)
ax.axhline(ZSCORE_THRESHOLD, color='#E53935', ls='--', lw=1.2, label=f'+{ZSCORE_THRESHOLD} threshold')
ax.axhline(-ZSCORE_THRESHOLD, color='#E53935', ls='--', lw=1.2, label=f'-{ZSCORE_THRESHOLD} threshold')
ax.axhline(0, color='gray', lw=0.5)

# Color the F4 pass points
f4p = df[df['f4_pass']]
if len(f4p) > 0:
    ax.scatter(f4p['tick'], f4p['zscore'], c='#4CAF50', s=15, zorder=5, label='F4 PASS', alpha=0.8)

# Color the F3-pass but F4-killed points
f4k = df[df['f3_pass'] & ~df['f4_pass']]
if len(f4k) > 0:
    ax.scatter(f4k['tick'], f4k['zscore'], c='#F44336', s=15, zorder=5, label='F4 KILLED', alpha=0.8)

ax.set_title('Z-Score Over Time — Points Outside Bands Are Statistically Unusual',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Z-Score')
ax.legend(fontsize=9, loc='upper right')

# ── Panel 3: Z-score distribution ──
ax = axes[2]
# All ticks z-scores (where we have data)
valid_z = df[df['rolling_std'].notna()]['zscore']
ax.hist(valid_z, bins=80, color='#B0BEC5', edgecolor='white', alpha=0.7, label='All ticks')

if len(f4p) > 0:
    ax.hist(f4p['zscore'], bins=20, color='#4CAF50', edgecolor='white', alpha=0.85, label='F4 Passed')

ax.axvline(ZSCORE_THRESHOLD, color='#E53935', ls='--', lw=1.5)
ax.axvline(-ZSCORE_THRESHOLD, color='#E53935', ls='--', lw=1.5)
ax.set_title('Z-Score Distribution — Passed Ticks Live in the Tails', fontsize=13, fontweight='bold')
ax.set_xlabel('Z-Score')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

# ── Panel 4: Progressive filter signals ──
ax = axes[3]
ax.fill_between(df['tick'], df['f1_pass'].astype(float), step='mid', alpha=0.15, color='#FF9800', label='F1')
ax.fill_between(df['tick'], df['f2_pass'].astype(float)*0.95, step='mid', alpha=0.2, color='#2196F3', label='F2')
ax.fill_between(df['tick'], df['f3_pass'].astype(float)*0.9, step='mid', alpha=0.3, color='#9C27B0', label='F3')
ax.fill_between(df['tick'], df['f4_pass'].astype(float)*0.85, step='mid', alpha=0.6, color='#4CAF50', label='F4')
ax.set_title('Progressive Filtering: F1 → F2 → F3 → F4', fontsize=13, fontweight='bold')
ax.set_ylabel('Signal')
ax.set_xlabel('Tick')
ax.set_yticks([0, 1])
ax.set_yticklabels(['DISCARD', 'PASS'])
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

---
## 23. Z-Score Sensitivity

In [ ]:
z_thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
z_sens = []

for zt in z_thresholds:
    zf = ZScoreFilter(lookback=ZSCORE_LOOKBACK, z_threshold=zt)
    test_df = zf.apply(df, exch_names)
    # F4 depends on f3_pass which is already in df
    n_pass = test_df['f4_pass'].sum()
    if n_pass > 0:
        p = test_df[test_df['f4_pass']]
        z_sens.append({
            'z_threshold': zt,
            'ticks_passed': n_pass,
            'pct_of_f3': round(n_pass / df['f3_pass'].sum() * 100, 1) if df['f3_pass'].sum() > 0 else 0,
            'avg_net_spread': round(p['net_spread_bps'].mean(), 1),
            'avg_zscore': round(p['zscore_abs'].mean(), 2),
        })
    else:
        z_sens.append({'z_threshold': zt, 'ticks_passed': 0, 'pct_of_f3': 0,
                       'avg_net_spread': 0, 'avg_zscore': 0})

zs_df = pd.DataFrame(z_sens)
print('Z-SCORE THRESHOLD SENSITIVITY')
print(zs_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(zs_df['z_threshold'], zs_df['ticks_passed'], 'o-', color='#2196F3', lw=2, ms=7)
ax.axvline(ZSCORE_THRESHOLD, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: |z| > {ZSCORE_THRESHOLD}')
ax.set_xlabel('Z-Score Threshold')
ax.set_ylabel('Ticks Passed')
ax.set_title('Signals vs Z-Score Threshold', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(zs_df['z_threshold'], zs_df['avg_net_spread'], 's-', color='#4CAF50', lw=2, ms=7)
ax.axvline(ZSCORE_THRESHOLD, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: |z| > {ZSCORE_THRESHOLD}')
ax.set_xlabel('Z-Score Threshold')
ax.set_ylabel('Avg Net Spread (bps)')
ax.set_title('Signal Quality vs Z-Score Threshold', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(' Higher z-threshold = only the most extreme spreads survive.')
print('  z=2 catches ~5% tail events. z=3 catches ~0.3% events.')

---
## 24. Combined Funnel — Filters 1 + 2 + 3 + 4

In [ ]:
total = len(df)
f1_p = df['f1_pass'].sum()
f2_p = df['f2_pass'].sum()
f3_p = df['f3_pass'].sum()
f4_p = df['f4_pass'].sum()

print('SIGNAL FUNNEL')
print('═' * 55)
print(f'  All ticks             : {total:>8,}')
print(f'  ↓ Filter 1 removed    : {total - f1_p:>8,}')
print(f'  After Filter 1        : {f1_p:>8,} ({f1_p/total*100:.1f}%)')
print(f'  ↓ Filter 2 removed    : {f1_p - f2_p:>8,}')
print(f'  After Filter 1+2      : {f2_p:>8,} ({f2_p/total*100:.1f}%)')
print(f'  ↓ Filter 3 removed    : {f2_p - f3_p:>8,}')
print(f'  After Filter 1+2+3    : {f3_p:>8,} ({f3_p/total*100:.1f}%)')
print(f'  ↓ Filter 4 removed    : {f3_p - f4_p:>8,}')
print(f'  After Filter 1+2+3+4  : {f4_p:>8,} ({f4_p/total*100:.1f}%)')
print('═' * 55)

# Funnel chart
fig, ax = plt.subplots(figsize=(12, 5))
stages = ['All Ticks', 'After F1\nspread>thresh', 'After F2\nnet>0', 'After F3\npersists 3+', 'After F4\n|z|>2']
counts = [total, f1_p, f2_p, f3_p, f4_p]
bar_colors = ['#90CAF9', '#FF9800', '#2196F3', '#9C27B0', '#4CAF50']

bars = ax.bar(stages, counts, color=bar_colors, edgecolor='white', width=0.55)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}', ha='center', fontsize=12, fontweight='bold')
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f'{count/total*100:.1f}%', ha='center', fontsize=10, color='white', fontweight='bold')

ax.set_title('Signal Funnel: Filter 1 → 2 → 3 → 4', fontsize=14, fontweight='bold')
ax.set_ylabel('Ticks Remaining')
ax.set_ylim(0, total * 1.15)
plt.tight_layout()
plt.show()

---
## 25. Export Updated Data

In [ ]:
export_cols = ['tick', 'true_price', 'Alpha', 'Beta', 'Gamma',
               'spread_bps', 'cost_bps', 'net_spread_bps',
               'best_buy', 'best_sell', 'streak',
               'rolling_mean', 'rolling_std', 'zscore', 'zscore_abs',
               'f1_pass', 'f2_pass', 'f3_pass', 'f4_pass']

df[export_cols].to_csv('filter1_2_3_4_full.csv', index=False)
df[df['f4_pass']][export_cols].to_csv('filter1_2_3_4_passed.csv', index=False)

print(f' Saved filter1_2_3_4_full.csv   — {len(df):,} rows (all ticks, all filter columns)')
print(f' Saved filter1_2_3_4_passed.csv — {f4_p:,} rows (passed all 4 filters)')
print(f'\n   → These go into Filter 5 (Vol-Adjusted Check) next.')

---
---

# 🔹 Filter 5 — Volatility-Adjusted Check
### Arbitrage Signal Pipeline: Step 5 of 5 (Final Filter)

---

**What this filter does:**

A 10 bps spread during a calm market is a real signal. The same 10 bps during a volatile open is just noise. This filter normalizes the spread by current market volatility:

$$\text{vol\_adjusted} = \frac{\text{spread}_{\text{bps}}}{\text{realized\_vol}_{\text{bps}}}$$

where realized vol is computed from recent true-price returns.

High ratio → spread is large relative to how much the market is moving → **real signal**

Low ratio → spread is just within normal market fluctuation → **noise**

---

---
## 26. Build Filter 5 — Volatility-Adjusted Check

In [ ]:
class VolAdjustedFilter:
    """
    Filter 5: Volatility-Adjusted Spread Check.

    Computes realized volatility from recent true-price returns,
    converts to bps, and checks if the spread is large relative to it.

    vol_adj_spread = spread_bps / realized_vol_bps

    If vol_adj_spread > threshold → PASS (spread is meaningful)
    If vol_adj_spread ≤ threshold → KILL (spread is just vol noise)

    Why this matters:
      During high-vol periods, exchange prices diverge naturally —
      the same 10 bps spread that's a great signal in a calm market
      is meaningless when the asset is swinging 50 bps per tick.

      This filter answers: 'Is this spread big RELATIVE to current conditions?'
    """

    def __init__(self, vol_lookback: int = 30, vol_adj_threshold: float = 1.5):
        self.vol_lookback = vol_lookback
        self.threshold = vol_adj_threshold

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Adds columns:
          - realized_vol_bps  : rolling realized vol of true_price in bps
          - vol_adj_spread    : spread_bps / realized_vol_bps
          - vol_regime        : 'low' / 'normal' / 'high' based on vol percentile
          - f5_pass           : True if f4_pass AND vol_adj_spread > threshold
        """
        result = df.copy()

        # ── Compute realized volatility ──
        # Returns in bps
        true_p = result['true_price'].values
        returns_bps = np.zeros(len(true_p))
        returns_bps[1:] = np.diff(np.log(true_p)) * 10000  # log returns in bps

        # Rolling std of returns = realized vol
        n = len(true_p)
        realized_vol = np.full(n, np.nan)

        for i in range(self.vol_lookback, n):
            window = returns_bps[i - self.vol_lookback + 1 : i + 1]
            realized_vol[i] = window.std()

        result['realized_vol_bps'] = realized_vol

        # ── Vol-adjusted spread ──
        result['vol_adj_spread'] = np.where(
            result['realized_vol_bps'] > 0.1,  # avoid division by near-zero
            result['spread_bps'] / result['realized_vol_bps'],
            0.0
        )

        # ── Vol regime classification ──
        vol_valid = result['realized_vol_bps'].dropna()
        if len(vol_valid) > 0:
            p33 = vol_valid.quantile(0.33)
            p66 = vol_valid.quantile(0.66)
            result['vol_regime'] = pd.cut(
                result['realized_vol_bps'],
                bins=[-np.inf, p33, p66, np.inf],
                labels=['low', 'normal', 'high']
            )
        else:
            result['vol_regime'] = 'normal'

        # ── Apply filter ──
        result['f5_pass'] = (
            result['f4_pass'] &
            (result['vol_adj_spread'] > self.threshold) &
            result['realized_vol_bps'].notna()
        )

        return result

    def summary(self, df: pd.DataFrame):
        """Print filter performance summary."""
        f4_p = df['f4_pass'].sum()
        f5_p = df['f5_pass'].sum()
        killed = f4_p - f5_p

        print('═' * 60)
        print('     FILTER 5 — VOLATILITY-ADJUSTED CHECK (FINAL FILTER)')
        print('═' * 60)
        print(f'  Vol lookback        : {self.vol_lookback} ticks')
        print(f'  Vol-adj threshold   : spread/vol > {self.threshold}')
        print(f'  Input (from F4)     : {f4_p:>10,} ticks')
        print(f'  PASSED              : {f5_p:>10,} ({f5_p/f4_p*100:.1f}% of F4)' if f4_p > 0 else '  PASSED: 0')
        print(f'  KILLED (vol noise)  : {killed:>10,} ({killed/f4_p*100:.1f}% of F4)' if f4_p > 0 else '  KILLED: 0')
        print('─' * 60)

        if f5_p > 0:
            p = df[df['f5_pass']]
            print(f'     FINAL TRADEABLE SIGNALS:')
            print(f'    Net spread       : mean={p["net_spread_bps"].mean():.1f}, max={p["net_spread_bps"].max():.1f} bps')
            print(f'    Vol-adj spread   : mean={p["vol_adj_spread"].mean():.2f}, max={p["vol_adj_spread"].max():.2f}')
            print(f'    |z-score|        : mean={p["zscore_abs"].mean():.2f}')
            print(f'    Streak           : mean={p["streak"].mean():.1f} ticks')
            print(f'    Realized vol     : mean={p["realized_vol_bps"].mean():.2f} bps')

        if killed > 0:
            k = df[df['f4_pass'] & ~df['f5_pass']]
            print(f'\n  Killed tick stats (spread was just vol noise):')
            print(f'    Net spread       : mean={k["net_spread_bps"].mean():.1f} bps')
            print(f'    Vol-adj spread   : mean={k["vol_adj_spread"].mean():.2f}')
            print(f'    Realized vol     : mean={k["realized_vol_bps"].mean():.2f} bps')
            print(f'    → Spread looked ok, but vol was high — it was noise')

        print('═' * 60)


print('VolAdjustedFilter class defined.')

---
## 27. Apply Filter 5

In [ ]:
VOL_LOOKBACK = 30
VOL_ADJ_THRESHOLD = 1.5

f5 = VolAdjustedFilter(vol_lookback=VOL_LOOKBACK, vol_adj_threshold=VOL_ADJ_THRESHOLD)
df = f5.apply(df)
f5.summary(df)

In [ ]:
# ── Show killed vs passed ──
killed_f5 = df[df['f4_pass'] & ~df['f5_pass']]
passed_f5 = df[df['f5_pass']]

print(f' KILLED by vol-adjustment ({len(killed_f5)} ticks):')
print('   These had a good spread, but market was too volatile — spread was noise.\n')
if len(killed_f5) > 0:
    display(killed_f5[['tick','spread_bps','net_spread_bps','vol_adj_spread',
                        'realized_vol_bps','vol_regime','zscore','best_buy','best_sell']].head(15))

print(f'\n FINAL TRADEABLE SIGNALS ({len(passed_f5)} ticks):')
print('   These survived ALL 5 filters — genuine, cost-viable, persistent, unusual, vol-meaningful.\n')
if len(passed_f5) > 0:
    display(passed_f5[['tick','spread_bps','net_spread_bps','vol_adj_spread',
                        'realized_vol_bps','vol_regime','zscore','streak',
                        'best_buy','best_sell']].head(20))

---
## 28. Visualize Filter 5 & Complete Pipeline

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(18, 20), gridspec_kw={'height_ratios': [2.5, 2, 2, 1.5]})

colors_exch = {'Alpha': '#2196F3', 'Beta': '#F44336', 'Gamma': '#4CAF50'}

# ── Panel 1: Prices with FINAL signals highlighted ──
ax = axes[0]
ax.plot(df['tick'], df['true_price'], '--', color='black', alpha=0.3, lw=0.7, label='True Price')
for name in exch_names:
    ax.plot(df['tick'], df[name], color=colors_exch[name], alpha=0.8, lw=0.6, label=name)

for _, row in df[df['f5_pass']].iterrows():
    ax.axvspan(row['tick'] - 0.5, row['tick'] + 0.5, alpha=0.2, color='#4CAF50')

ax.set_title('Exchange Prices — Green = FINAL TRADEABLE SIGNALS (passed all 5 filters)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)

# ── Panel 2: Vol-adjusted spread with threshold ──
ax = axes[1]
valid = df[df['realized_vol_bps'].notna()]
ax.plot(valid['tick'], valid['vol_adj_spread'], color='#607D8B', alpha=0.4, lw=0.5)

# Color F4 pass ticks
f4_ticks = df[df['f4_pass'] & df['realized_vol_bps'].notna()]
if len(f4_ticks) > 0:
    colors_scatter = ['#4CAF50' if p else '#F44336' for p in f4_ticks['f5_pass']]
    ax.scatter(f4_ticks['tick'], f4_ticks['vol_adj_spread'], c=colors_scatter,
               s=20, zorder=5, alpha=0.8)

ax.axhline(VOL_ADJ_THRESHOLD, color='black', ls='--', lw=1.5,
           label=f'Threshold: {VOL_ADJ_THRESHOLD}')
ax.set_title('Vol-Adjusted Spread — Green = PASS, Red = KILLED by vol',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Spread / Realized Vol')
ax.set_ylim(0, min(valid['vol_adj_spread'].quantile(0.995) * 1.3, 30))
ax.legend(fontsize=10)

# ── Panel 3: Realized vol over time with regime coloring ──
ax = axes[2]
vol_data = df[df['realized_vol_bps'].notna()]
regime_colors = {'low': '#4CAF50', 'normal': '#FF9800', 'high': '#F44336'}

ax.fill_between(vol_data['tick'], vol_data['realized_vol_bps'], alpha=0.4, color='#90CAF9')
ax.plot(vol_data['tick'], vol_data['realized_vol_bps'], color='#1565C0', lw=0.6)

# Mark vol regimes
vol_vals = vol_data['realized_vol_bps'].dropna()
if len(vol_vals) > 0:
    p33 = vol_vals.quantile(0.33)
    p66 = vol_vals.quantile(0.66)
    ax.axhline(p33, color='#4CAF50', ls=':', lw=1, alpha=0.7, label=f'Low/Normal: {p33:.1f}')
    ax.axhline(p66, color='#F44336', ls=':', lw=1, alpha=0.7, label=f'Normal/High: {p66:.1f}')

ax.set_title('Realized Volatility (bps) — Market Regime Over Time',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Realized Vol (bps)')
ax.legend(fontsize=9)

# ── Panel 4: All 5 filters progressive ──
ax = axes[3]
ax.fill_between(df['tick'], df['f1_pass'].astype(float), step='mid', alpha=0.10, color='#FF9800', label='F1')
ax.fill_between(df['tick'], df['f2_pass'].astype(float)*0.95, step='mid', alpha=0.15, color='#2196F3', label='F2')
ax.fill_between(df['tick'], df['f3_pass'].astype(float)*0.9, step='mid', alpha=0.2, color='#9C27B0', label='F3')
ax.fill_between(df['tick'], df['f4_pass'].astype(float)*0.85, step='mid', alpha=0.3, color='#FF5722', label='F4')
ax.fill_between(df['tick'], df['f5_pass'].astype(float)*0.8, step='mid', alpha=0.7, color='#4CAF50', label='F5 ✅')
ax.set_title('Complete Filter Pipeline: F1 → F2 → F3 → F4 → F5', fontsize=13, fontweight='bold')
ax.set_ylabel('Signal')
ax.set_xlabel('Tick')
ax.set_yticks([0, 1])
ax.set_yticklabels(['DISCARD', 'PASS'])
ax.legend(fontsize=9, loc='upper right', ncol=5)

plt.tight_layout()
plt.show()

---
## 29. Vol-Adjusted Sensitivity

In [ ]:
va_thresholds = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
va_sens = []

for vt in va_thresholds:
    vf = VolAdjustedFilter(vol_lookback=VOL_LOOKBACK, vol_adj_threshold=vt)
    test_df = vf.apply(df)
    n_pass = test_df['f5_pass'].sum()
    if n_pass > 0:
        p = test_df[test_df['f5_pass']]
        va_sens.append({
            'vol_adj_threshold': vt,
            'ticks_passed': n_pass,
            'pct_of_f4': round(n_pass / max(df['f4_pass'].sum(), 1) * 100, 1),
            'avg_net_spread': round(p['net_spread_bps'].mean(), 1),
            'avg_vol_adj': round(p['vol_adj_spread'].mean(), 2),
        })
    else:
        va_sens.append({'vol_adj_threshold': vt, 'ticks_passed': 0,
                        'pct_of_f4': 0, 'avg_net_spread': 0, 'avg_vol_adj': 0})

va_df = pd.DataFrame(va_sens)
print('VOL-ADJUSTED THRESHOLD SENSITIVITY')
print(va_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(va_df['vol_adj_threshold'], va_df['ticks_passed'], 'o-', color='#2196F3', lw=2, ms=7)
ax.axvline(VOL_ADJ_THRESHOLD, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: {VOL_ADJ_THRESHOLD}')
ax.set_xlabel('Vol-Adjusted Threshold')
ax.set_ylabel('Ticks Passed')
ax.set_title('Signals vs Vol-Adj Threshold', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(va_df['vol_adj_threshold'], va_df['avg_net_spread'], 's-', color='#4CAF50', lw=2, ms=7)
ax.axvline(VOL_ADJ_THRESHOLD, color='red', ls='--', lw=1, alpha=0.7,
           label=f'Our choice: {VOL_ADJ_THRESHOLD}')
ax.set_xlabel('Vol-Adjusted Threshold')
ax.set_ylabel('Avg Net Spread (bps)')
ax.set_title('Signal Quality vs Vol-Adj Threshold', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 30. FINAL FUNNEL — All 5 Filters

In [ ]:
total = len(df)
f1_p = df['f1_pass'].sum()
f2_p = df['f2_pass'].sum()
f3_p = df['f3_pass'].sum()
f4_p = df['f4_pass'].sum()
f5_p = df['f5_pass'].sum()

print('🔬 COMPLETE SIGNAL FUNNEL')
print('═' * 60)
print(f'  All ticks               : {total:>8,}')
print(f'  ↓ F1: Spread > threshold: -{total - f1_p:>7,}  →  {f1_p:>6,} remain ({f1_p/total*100:.1f}%)')
print(f'  ↓ F2: Net after costs   : -{f1_p - f2_p:>7,}  →  {f2_p:>6,} remain ({f2_p/total*100:.1f}%)')
print(f'  ↓ F3: Persists 3+ ticks : -{f2_p - f3_p:>7,}  →  {f3_p:>6,} remain ({f3_p/total*100:.1f}%)')
print(f'  ↓ F4: |z-score| > 2     : -{f3_p - f4_p:>7,}  →  {f4_p:>6,} remain ({f4_p/total*100:.1f}%)')
print(f'  ↓ F5: Vol-adjusted > 1.5: -{f4_p - f5_p:>7,}  →  {f5_p:>6,} remain ({f5_p/total*100:.1f}%)')
print(f'  ─────────────────────────────────────────────────────')
print(f'    TRADEABLE SIGNALS     :           {f5_p:>6,} ({f5_p/total*100:.1f}% of all ticks)')
print('═' * 60)

# Funnel chart
fig, ax = plt.subplots(figsize=(14, 6))
stages = ['All Ticks\n(raw)', 'F1\nspread>5bps', 'F2\nnet>0', 'F3\npersists 3+', 'F4\n|z|>2', 'F5 ✅\nvol-adj>1.5']
counts = [total, f1_p, f2_p, f3_p, f4_p, f5_p]
bar_colors = ['#90CAF9', '#FF9800', '#2196F3', '#9C27B0', '#FF5722', '#4CAF50']

bars = ax.bar(stages, counts, color=bar_colors, edgecolor='white', width=0.55)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}', ha='center', fontsize=12, fontweight='bold')
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f'{count/total*100:.1f}%', ha='center', fontsize=10, color='white', fontweight='bold')

# Add arrows between bars
for i in range(len(counts)-1):
    if counts[i] > 0 and counts[i+1] > 0:
        killed = counts[i] - counts[i+1]
        mid_x = (bars[i].get_x() + bars[i].get_width() + bars[i+1].get_x()) / 2
        mid_y = max(counts[i], counts[i+1]) * 0.5

ax.set_title('Complete Signal Funnel — 5 Filters', fontsize=15, fontweight='bold')
ax.set_ylabel('Ticks Remaining')
ax.set_ylim(0, total * 1.15)
plt.tight_layout()
plt.show()

---
## 31. Final Export — Tradeable Signals

In [ ]:
export_cols = ['tick', 'true_price', 'Alpha', 'Beta', 'Gamma',
               'spread_bps', 'cost_bps', 'net_spread_bps',
               'best_buy', 'best_sell', 'streak',
               'rolling_mean', 'rolling_std', 'zscore', 'zscore_abs',
               'realized_vol_bps', 'vol_adj_spread', 'vol_regime',
               'f1_pass', 'f2_pass', 'f3_pass', 'f4_pass', 'f5_pass']

df[export_cols].to_csv('complete_pipeline_full.csv', index=False)
df[df['f5_pass']][export_cols].to_csv('tradeable_signals.csv', index=False)

print(f' Saved complete_pipeline_full.csv — {len(df):,} rows (all ticks, all 5 filter columns)')
print(f' Saved tradeable_signals.csv      — {f5_p:,} rows (FINAL tradeable signals)')
print(f'\n Pipeline complete. {f5_p} tradeable signals ready for execution.')
print(f'   Next step: feed these into the arbitrage engine for backtesting!')

---
---

# 🔹 Backtest Engine
### Simulating Actual Trades on Filtered Signals

---

Now we execute trades on the 5-filter signals and compute realized P&L.

**Rules:**
- Entry at signal tick + 1 (latency)
- Group consecutive signals on same pair into one trade
- Exit on: z-score reversion | stop-loss | max hold time
- Track full P&L after fees

---

---
## 32. Backtest Configuration

In [ ]:
# ── Backtest Parameters ──
INITIAL_CAPITAL = 1_000_000.0    # $1M starting capital
TRADE_NOTIONAL = 100_000.0        # $50K per trade
LATENCY_TICKS = 1                # Enter 1 tick after signal
MAX_HOLD_TICKS = 20              # Force exit after 20 ticks
STOP_LOSS_BPS = 15.0             # Exit if spread widens 15 bps against us
ZSCORE_EXIT = 0.5                # Exit when z-score reverts below this

# Fees already defined above:
# EXCHANGE_FEES_BPS = {'Alpha': 2.0, 'Beta': 3.0, 'Gamma': 1.5}
# SLIPPAGE_BPS = 1.0

print('   BACKTEST CONFIG')
print(f'   Capital         : ${INITIAL_CAPITAL:,.0f}')
print(f'   Trade size      : ${TRADE_NOTIONAL:,.0f}')
print(f'   Latency         : {LATENCY_TICKS} tick(s)')
print(f'   Max hold        : {MAX_HOLD_TICKS} ticks')
print(f'   Stop-loss       : {STOP_LOSS_BPS} bps widening')
print(f'   Z-score exit    : |z| < {ZSCORE_EXIT}')

---
## 33. Backtest Engine

In [ ]:
class BacktestEngine:
    """
    Executes trades on filtered signals and computes realized P&L.

    Logic:
      1. Walk through filtered signals
      2. Group consecutive signals on same pair → one trade entry
      3. Enter at signal_tick + latency
      4. Each tick, check exit conditions:
         a) Z-score reversion (spread normalized) → take profit
         b) Stop-loss (spread widened against us) → cut loss
         c) Max hold time reached → forced exit
      5. Compute P&L for each closed trade
    """

    def __init__(self, capital, notional, latency, max_hold,
                 stop_loss_bps, zscore_exit, fees_bps, slippage_bps):
        self.capital = capital
        self.initial_capital = capital
        self.notional = notional
        self.latency = latency
        self.max_hold = max_hold
        self.stop_loss = stop_loss_bps
        self.z_exit = zscore_exit
        self.fees = fees_bps
        self.slippage = slippage_bps
        self.trades = []
        self.equity_curve = []

    def _round_trip_cost_bps(self, buy_exch, sell_exch):
        """Total cost for entry + exit on both legs."""
        return (self.fees[buy_exch] + self.fees[sell_exch]) * 2 + self.slippage * 4

    def _get_spread_bps(self, row, buy_exch, sell_exch):
        """Compute spread in bps at a given row."""
        bp = row[buy_exch]
        sp = row[sell_exch]
        mid = (bp + sp) / 2
        return (sp - bp) / mid * 10000

    def run(self, df, signals_df):
        """
        df: full price data (all 2000 ticks)
        signals_df: only f5_pass=True rows

        Returns: trades list, equity curve
        """
        self.trades = []
        self.capital = self.initial_capital
        self.equity_curve = [{'tick': 0, 'capital': self.capital, 'n_open': 0}]

        N = len(df)
        signal_ticks = signals_df['tick'].values

        # ── Group consecutive signals on same pair ──
        # Only enter on the FIRST tick of each cluster
        entry_signals = []
        prev_tick = -999
        prev_pair = ''

        for _, sig in signals_df.iterrows():
            pair = f"{sig['best_buy']}|{sig['best_sell']}"
            tick = int(sig['tick'])

            # New cluster if: different pair OR gap > 2 ticks from last signal
            if pair != prev_pair or (tick - prev_tick) > 2:
                entry_signals.append(sig)

            prev_tick = tick
            prev_pair = pair

        print(f'📊 {len(signals_df)} filtered signals → {len(entry_signals)} unique trade entries')
        print(f'   (consecutive signals on same pair grouped into one trade)\n')

        # ── Execute each trade ──
        occupied_until = -1  # prevents overlapping trades

        for sig in entry_signals:
            signal_tick = int(sig['tick'])
            entry_tick = signal_tick + self.latency

            if entry_tick >= N:
                continue
            if entry_tick <= occupied_until:
                continue  # still in a previous trade

            buy_exch = sig['best_buy']
            sell_exch = sig['best_sell']
            entry_row = df.iloc[entry_tick]

            entry_buy_price = entry_row[buy_exch]
            entry_sell_price = entry_row[sell_exch]
            entry_spread = self._get_spread_bps(entry_row, buy_exch, sell_exch)

            # ── Hold and monitor for exit ──
            exit_tick = None
            exit_reason = None

            for t in range(entry_tick + 1, min(entry_tick + self.max_hold + 1, N)):
                row_t = df.iloc[t]
                current_spread = self._get_spread_bps(row_t, buy_exch, sell_exch)

                # ── Exit condition A: Z-score reversion ──
                if 'zscore_abs' in df.columns and not np.isnan(row_t.get('zscore_abs', np.nan)):
                    if row_t['zscore_abs'] < self.z_exit:
                        exit_tick = t
                        exit_reason = 'zscore_revert'
                        break

                # ── Exit condition B: Stop-loss ──
                spread_change = current_spread - entry_spread
                if spread_change > self.stop_loss:  # spread widened against us
                    exit_tick = t
                    exit_reason = 'stop_loss'
                    break

            # ── Exit condition C: Max hold ──
            if exit_tick is None:
                exit_tick = min(entry_tick + self.max_hold, N - 1)
                exit_reason = 'max_hold'

            # ── Compute P&L ──
            exit_row = df.iloc[exit_tick]
            exit_buy_price = exit_row[buy_exch]
            exit_sell_price = exit_row[sell_exch]

            qty = self.notional / entry_buy_price

            # Long leg: bought at entry_buy, now worth exit_buy
            long_pnl = qty * (exit_buy_price - entry_buy_price)
            # Short leg: sold at entry_sell, now must buy back at exit_sell
            short_pnl = qty * (entry_sell_price - exit_sell_price)
            gross_pnl = long_pnl + short_pnl

            # Fees: entry + exit on both legs
            total_cost = self.notional * self._round_trip_cost_bps(buy_exch, sell_exch) / 10000
            net_pnl = gross_pnl - total_cost

            self.capital += net_pnl
            occupied_until = exit_tick

            self.trades.append({
                'signal_tick': signal_tick,
                'entry_tick': entry_tick,
                'exit_tick': exit_tick,
                'hold_ticks': exit_tick - entry_tick,
                'buy_exch': buy_exch,
                'sell_exch': sell_exch,
                'entry_buy_p': round(entry_buy_price, 4),
                'entry_sell_p': round(entry_sell_price, 4),
                'exit_buy_p': round(exit_buy_price, 4),
                'exit_sell_p': round(exit_sell_price, 4),
                'entry_spread_bps': round(entry_spread, 2),
                'exit_spread_bps': round(self._get_spread_bps(exit_row, buy_exch, sell_exch), 2),
                'gross_pnl': round(gross_pnl, 2),
                'fees': round(total_cost, 2),
                'net_pnl': round(net_pnl, 2),
                'exit_reason': exit_reason,
                'capital_after': round(self.capital, 2),
            })

            self.equity_curve.append({
                'tick': exit_tick,
                'capital': round(self.capital, 2),
                'n_open': 0
            })

        return pd.DataFrame(self.trades), pd.DataFrame(self.equity_curve)


print('BacktestEngine defined.')

---
## 34. Run Backtest

In [ ]:
signals = df[df['f5_pass']].copy()

bt = BacktestEngine(
    capital=INITIAL_CAPITAL,
    notional=TRADE_NOTIONAL,
    latency=LATENCY_TICKS,
    max_hold=MAX_HOLD_TICKS,
    stop_loss_bps=STOP_LOSS_BPS,
    zscore_exit=ZSCORE_EXIT,
    fees_bps=EXCHANGE_FEES_BPS,
    slippage_bps=SLIPPAGE_BPS,
)

trades_df, equity_df = bt.run(df, signals)

print(f'\n Backtest complete: {len(trades_df)} trades executed.')

In [ ]:
# Show all trades
if not trades_df.empty:
    print('TRADE LOG\n')
    display(trades_df)

---
## 35. Performance Report

In [ ]:
def performance_report(trades_df, equity_df, initial_capital):
    if trades_df.empty:
        print('⚠️ No trades to analyze.')
        return

    pnl = trades_df['net_pnl']
    n = len(pnl)
    winners = pnl[pnl > 0]
    losers = pnl[pnl <= 0]
    total_pnl = pnl.sum()

    # Win rate
    win_rate = len(winners) / n * 100 if n > 0 else 0

    # Profit factor
    gross_profit = winners.sum() if len(winners) > 0 else 0
    gross_loss = abs(losers.sum()) if len(losers) > 0 else 1
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

    # Max drawdown
    cap = equity_df['capital'].values
    peak = np.maximum.accumulate(cap)
    dd = (peak - cap) / peak
    max_dd = dd.max() * 100

    # Avg hold
    avg_hold = trades_df['hold_ticks'].mean()

    # Return
    total_return = total_pnl / initial_capital * 100
    final_capital = cap[-1] if len(cap) > 0 else initial_capital

    # Total fees
    total_fees = trades_df['fees'].sum()

    print('═' * 60)
    print('            BACKTEST PERFORMANCE REPORT')
    print('═' * 60)
    print(f'  Initial Capital     : ${initial_capital:>14,.2f}')
    print(f'  Final Capital       : ${final_capital:>14,.2f}')
    print(f'  Total Net P&L       : ${total_pnl:>14,.2f}')
    print(f'  Return              : {total_return:>13.3f}%')
    print(f'  Total Fees Paid     : ${total_fees:>14,.2f}')
    print('─' * 60)
    print(f'  Total Trades        : {n:>14}')
    print(f'  Winners             : {len(winners):>14} ({win_rate:.1f}%)')
    print(f'  Losers              : {len(losers):>14} ({100-win_rate:.1f}%)')
    print(f'  Avg Win             : ${winners.mean():>14,.2f}' if len(winners) > 0 else '  Avg Win             :            N/A')
    print(f'  Avg Loss            : ${losers.mean():>14,.2f}' if len(losers) > 0 else '  Avg Loss            :            N/A')
    print(f'  Largest Win         : ${winners.max():>14,.2f}' if len(winners) > 0 else '')
    print(f'  Largest Loss        : ${losers.min():>14,.2f}' if len(losers) > 0 else '')
    print(f'  Profit Factor       : {profit_factor:>14.2f}')
    print(f'  Max Drawdown        : {max_dd:>13.3f}%')
    print(f'  Avg Hold Time       : {avg_hold:>13.1f} ticks')
    print('─' * 60)
    print(f'  EXIT REASONS:')
    for reason, count in trades_df['exit_reason'].value_counts().items():
        avg = trades_df[trades_df['exit_reason']==reason]['net_pnl'].mean()
        print(f'    {reason:<20s}: {count:>3} trades, avg P&L ${avg:>8,.2f}')
    print('─' * 60)
    print(f'  BY EXCHANGE PAIR:')
    pairs = trades_df.groupby(['buy_exch','sell_exch']).agg(
        count=('net_pnl','count'),
        total_pnl=('net_pnl','sum'),
        avg_pnl=('net_pnl','mean'),
        win_rate=('net_pnl', lambda x: (x>0).mean()*100)
    ).round(2)
    for (be, se), row in pairs.iterrows():
        print(f'    {be}→{se}: {int(row["count"]):>3} trades, '
              f'P&L ${row["total_pnl"]:>8,.2f}, '
              f'avg ${row["avg_pnl"]:>8,.2f}, '
              f'WR {row["win_rate"]:.0f}%')
    print('═' * 60)

performance_report(trades_df, equity_df, INITIAL_CAPITAL)

---
## 36. Visualize Backtest Results

In [ ]:
if not trades_df.empty:
    fig, axes = plt.subplots(4, 2, figsize=(18, 22), gridspec_kw={'height_ratios': [2.5, 2, 2, 2]})

    colors_exch = {'Alpha': '#2196F3', 'Beta': '#F44336', 'Gamma': '#4CAF50'}

    # ── 1. Equity Curve (full width) ──
    ax = axes[0, 0]
    cap = equity_df['capital'].values
    ax.plot(equity_df['tick'], cap, 'o-', color='#1565C0', lw=1.5, ms=4)
    ax.axhline(INITIAL_CAPITAL, color='gray', ls='--', alpha=0.5, label='Starting Capital')
    ax.fill_between(equity_df['tick'], INITIAL_CAPITAL, cap,
                     where=cap >= INITIAL_CAPITAL, alpha=0.15, color='green')
    ax.fill_between(equity_df['tick'], INITIAL_CAPITAL, cap,
                     where=cap < INITIAL_CAPITAL, alpha=0.15, color='red')
    ax.set_title('Equity Curve', fontsize=14, fontweight='bold')
    ax.set_ylabel('Capital ($)')
    ax.set_xlabel('Tick')
    ax.legend(fontsize=9)

    # ── 2. Prices with trade entry/exit markers ──
    ax = axes[0, 1]
    for name in exch_names:
        ax.plot(df['tick'], df[name], color=colors_exch[name], alpha=0.6, lw=0.5, label=name)
    for _, trade in trades_df.iterrows():
        ax.axvspan(trade['entry_tick'], trade['exit_tick'],
                   alpha=0.15, color='#4CAF50' if trade['net_pnl'] > 0 else '#F44336')
        ax.plot(trade['entry_tick'], trade['entry_buy_p'], '^', color='green', ms=6)
        ax.plot(trade['exit_tick'], trade['exit_buy_p'], 'v', color='red', ms=6)
    ax.set_title('Prices with Trade Windows (green=profit, red=loss)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.set_xlabel('Tick')
    ax.legend(fontsize=8, loc='upper left')

    # ── 3. P&L per trade ──
    ax = axes[1, 0]
    bar_colors = ['#4CAF50' if p > 0 else '#F44336' for p in trades_df['net_pnl']]
    ax.bar(range(len(trades_df)), trades_df['net_pnl'], color=bar_colors, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Net P&L Per Trade', fontsize=13, fontweight='bold')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Net P&L ($)')

    # ── 4. Cumulative P&L ──
    ax = axes[1, 1]
    cum_pnl = trades_df['net_pnl'].cumsum()
    ax.plot(range(len(cum_pnl)), cum_pnl, 'o-', color='#1565C0', lw=1.5, ms=4)
    ax.fill_between(range(len(cum_pnl)), 0, cum_pnl,
                     where=cum_pnl >= 0, alpha=0.15, color='green')
    ax.fill_between(range(len(cum_pnl)), 0, cum_pnl,
                     where=cum_pnl < 0, alpha=0.15, color='red')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title('Cumulative P&L', fontsize=13, fontweight='bold')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative P&L ($)')

    # ── 5. P&L Distribution ──
    ax = axes[2, 0]
    ax.hist(trades_df['net_pnl'], bins=max(10, len(trades_df)//3),
            color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(0, color='red', ls='--', lw=1.5)
    ax.axvline(trades_df['net_pnl'].mean(), color='#4CAF50', ls='--', lw=1.5,
               label=f'Mean: ${trades_df["net_pnl"].mean():.2f}')
    ax.set_title('P&L Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Net P&L ($)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

    # ── 6. Exit Reason Breakdown ──
    ax = axes[2, 1]
    exit_stats = trades_df.groupby('exit_reason')['net_pnl'].agg(['mean','count'])
    exit_colors = {'zscore_revert': '#4CAF50', 'stop_loss': '#F44336', 'max_hold': '#FF9800'}
    bc = [exit_colors.get(r, '#607D8B') for r in exit_stats.index]
    ax.bar(exit_stats.index, exit_stats['mean'], color=bc, edgecolor='white')
    for i, (reason, row) in enumerate(exit_stats.iterrows()):
        ax.text(i, row['mean'], f'n={int(row["count"])}', ha='center',
                va='bottom' if row['mean'] >= 0 else 'top', fontsize=10)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Avg P&L by Exit Reason', fontsize=13, fontweight='bold')
    ax.set_ylabel('Avg Net P&L ($)')

    # ── 7. Entry Spread vs Realized P&L ──
    ax = axes[3, 0]
    sc_colors = ['#4CAF50' if p > 0 else '#F44336' for p in trades_df['net_pnl']]
    ax.scatter(trades_df['entry_spread_bps'], trades_df['net_pnl'],
               c=sc_colors, s=40, alpha=0.7, edgecolors='white', lw=0.5)
    ax.axhline(0, color='gray', lw=0.8)
    # Trend line
    if len(trades_df) > 2:
        z = np.polyfit(trades_df['entry_spread_bps'], trades_df['net_pnl'], 1)
        xl = np.linspace(trades_df['entry_spread_bps'].min(), trades_df['entry_spread_bps'].max(), 50)
        ax.plot(xl, np.polyval(z, xl), 'b--', lw=1.5, alpha=0.6)
    ax.set_title('Entry Spread vs Realized P&L', fontsize=13, fontweight='bold')
    ax.set_xlabel('Entry Spread (bps)')
    ax.set_ylabel('Net P&L ($)')

    # ── 8. Hold Time vs P&L ──
    ax = axes[3, 1]
    ax.scatter(trades_df['hold_ticks'], trades_df['net_pnl'],
               c=sc_colors, s=40, alpha=0.7, edgecolors='white', lw=0.5)
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_title('Hold Time vs Realized P&L', fontsize=13, fontweight='bold')
    ax.set_xlabel('Hold Time (ticks)')
    ax.set_ylabel('Net P&L ($)')

    plt.tight_layout()
    plt.show()
    print('8-panel backtest dashboard rendered.')

In [ ]:
"""
Complete Backtest Performance Metrics Calculator
Paste this into any notebook after running the backtest.

Requires:
  - trades_df : DataFrame with columns ['net_pnl', 'fees', 'hold_ticks', 'exit_reason', 'gross_pnl']
  - equity_df : DataFrame with columns ['tick', 'capital']
  - INITIAL_CAPITAL : float
"""

import numpy as np
import pandas as pd


def compute_all_metrics(trades_df, equity_df, initial_capital, ticks_per_day=390):
    """
    Computes every relevant backtest ratio from trades and equity data.

    Returns a dict of all metrics + prints a formatted report.
    """

    if trades_df.empty:
        print('⚠️ No trades to analyze.')
        return {}

    pnl = trades_df['net_pnl']
    n = len(pnl)
    winners = pnl[pnl > 0]
    losers = pnl[pnl <= 0]
    total_pnl = pnl.sum()
    total_fees = trades_df['fees'].sum()
    final_capital = equity_df['capital'].iloc[-1]
    cap = equity_df['capital'].values

    # ─────────────────────────────────────────────────
    # 1. RETURN METRICS
    # ─────────────────────────────────────────────────

    total_return_pct = (final_capital - initial_capital) / initial_capital * 100

    # Approximate annualized return (assuming 252 trading days)
    n_ticks = equity_df['tick'].iloc[-1] - equity_df['tick'].iloc[0]
    n_days = max(n_ticks / ticks_per_day, 1)
    annualized_return = ((final_capital / initial_capital) ** (252 / n_days) - 1) * 100

    # ─────────────────────────────────────────────────
    # 2. RISK METRICS
    # ─────────────────────────────────────────────────

    # Max Drawdown
    peak = np.maximum.accumulate(cap)
    drawdown = (peak - cap) / peak
    max_drawdown_pct = drawdown.max() * 100

    # Max Drawdown Duration (ticks)
    in_dd = drawdown > 0
    dd_durations = []
    dd_start = None
    for i in range(len(in_dd)):
        if in_dd[i] and dd_start is None:
            dd_start = i
        elif not in_dd[i] and dd_start is not None:
            dd_durations.append(i - dd_start)
            dd_start = None
    if dd_start is not None:
        dd_durations.append(len(in_dd) - dd_start)
    max_dd_duration = max(dd_durations) if dd_durations else 0

    # ─────────────────────────────────────────────────
    # 3. RISK-ADJUSTED RETURN RATIOS
    # ─────────────────────────────────────────────────

    # Per-trade returns (as fraction of capital)
    trade_returns = pnl / initial_capital

    # Sharpe Ratio (annualized)
    # Annualize by sqrt(trades_per_year)
    trades_per_day = n / max(n_days, 1)
    trades_per_year = trades_per_day * 252
    if trade_returns.std() > 0:
        sharpe = (trade_returns.mean() / trade_returns.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = 0.0

    # Sortino Ratio (annualized) — only penalizes downside vol
    downside_returns = trade_returns[trade_returns < 0]
    if len(downside_returns) > 0 and downside_returns.std() > 0:
        sortino = (trade_returns.mean() / downside_returns.std()) * np.sqrt(trades_per_year)
    else:
        sortino = float('inf') if trade_returns.mean() > 0 else 0.0

    # Calmar Ratio — annualized return / max drawdown
    calmar = annualized_return / max_drawdown_pct if max_drawdown_pct > 0 else 0.0

    # ─────────────────────────────────────────────────
    # 4. WIN/LOSS METRICS
    # ─────────────────────────────────────────────────

    win_rate = len(winners) / n * 100
    loss_rate = 100 - win_rate

    avg_win = winners.mean() if len(winners) > 0 else 0
    avg_loss = losers.mean() if len(losers) > 0 else 0
    largest_win = winners.max() if len(winners) > 0 else 0
    largest_loss = losers.min() if len(losers) > 0 else 0

    # Profit Factor — gross wins / gross losses
    gross_profit = winners.sum() if len(winners) > 0 else 0
    gross_loss = abs(losers.sum()) if len(losers) > 0 else 1
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

    # Payoff Ratio — avg win / avg loss
    payoff_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')

    # Expectancy — expected $ per trade
    expectancy = (win_rate / 100 * avg_win) + (loss_rate / 100 * avg_loss)

    # Kelly Criterion — optimal fraction to bet
    # Kelly = W - (1-W)/R where W=win rate, R=payoff ratio
    if payoff_ratio > 0 and payoff_ratio != float('inf'):
        kelly = (win_rate / 100) - ((1 - win_rate / 100) / payoff_ratio)
    else:
        kelly = 0.0

    # ─────────────────────────────────────────────────
    # 5. CONSISTENCY METRICS
    # ─────────────────────────────────────────────────

    # Consecutive wins/losses
    streaks_win = []
    streaks_loss = []
    current_w = 0
    current_l = 0
    for p in pnl:
        if p > 0:
            current_w += 1
            if current_l > 0:
                streaks_loss.append(current_l)
            current_l = 0
        else:
            current_l += 1
            if current_w > 0:
                streaks_win.append(current_w)
            current_w = 0
    if current_w > 0:
        streaks_win.append(current_w)
    if current_l > 0:
        streaks_loss.append(current_l)

    max_consec_wins = max(streaks_win) if streaks_win else 0
    max_consec_losses = max(streaks_loss) if streaks_loss else 0

    # Skewness and Kurtosis of P&L
    skewness = pnl.skew()
    kurtosis = pnl.kurtosis()

    # ─────────────────────────────────────────────────
    # 6. EFFICIENCY METRICS
    # ─────────────────────────────────────────────────

    avg_hold = trades_df['hold_ticks'].mean()
    total_hold = trades_df['hold_ticks'].sum()
    time_in_market_pct = total_hold / max(n_ticks, 1) * 100

    # Return per tick in market
    return_per_tick = total_pnl / max(total_hold, 1)

    # Fee drag — fees as % of gross profit
    gross_total = trades_df['gross_pnl'].sum() if 'gross_pnl' in trades_df.columns else total_pnl + total_fees
    fee_drag_pct = total_fees / abs(gross_total) * 100 if gross_total != 0 else 0

    # ─────────────────────────────────────────────────
    # BUILD RESULTS DICT
    # ─────────────────────────────────────────────────

    metrics = {
        # Returns
        'total_pnl': total_pnl,
        'total_return_pct': total_return_pct,
        'annualized_return_pct': annualized_return,
        'final_capital': final_capital,

        # Risk
        'max_drawdown_pct': max_drawdown_pct,
        'max_dd_duration_ticks': max_dd_duration,

        # Risk-adjusted
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'calmar_ratio': calmar,

        # Win/loss
        'total_trades': n,
        'winners': len(winners),
        'losers': len(losers),
        'win_rate_pct': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'largest_win': largest_win,
        'largest_loss': largest_loss,
        'profit_factor': profit_factor,
        'payoff_ratio': payoff_ratio,
        'expectancy_per_trade': expectancy,
        'kelly_criterion': kelly,

        # Consistency
        'max_consec_wins': max_consec_wins,
        'max_consec_losses': max_consec_losses,
        'pnl_skewness': skewness,
        'pnl_kurtosis': kurtosis,

        # Efficiency
        'avg_hold_ticks': avg_hold,
        'time_in_market_pct': time_in_market_pct,
        'return_per_tick': return_per_tick,
        'total_fees': total_fees,
        'fee_drag_pct': fee_drag_pct,
    }

    # ─────────────────────────────────────────────────
    # PRINT REPORT
    # ─────────────────────────────────────────────────

    print('═' * 62)
    print('           COMPLETE BACKTEST PERFORMANCE REPORT')
    print('═' * 62)

    print('\n  ── RETURNS ──')
    print(f'    Initial Capital        : ${initial_capital:>14,.2f}')
    print(f'    Final Capital          : ${final_capital:>14,.2f}')
    print(f'    Total Net P&L          : ${total_pnl:>14,.2f}')
    print(f'    Total Return           : {total_return_pct:>13.3f}%')
    print(f'    Annualized Return      : {annualized_return:>13.2f}%')

    print('\n  ── RISK ──')
    print(f'    Max Drawdown           : {max_drawdown_pct:>13.4f}%')
    print(f'    Max DD Duration        : {max_dd_duration:>13} ticks')

    print('\n  ── RISK-ADJUSTED RATIOS ──')
    print(f'    Sharpe Ratio (ann.)    : {sharpe:>13.2f}')
    print(f'    Sortino Ratio (ann.)   : {sortino:>13.2f}')
    print(f'    Calmar Ratio           : {calmar:>13.2f}')

    print('\n  ── WIN/LOSS ──')
    print(f'    Total Trades           : {n:>13}')
    print(f'    Win Rate               : {win_rate:>12.1f}%')
    print(f'    Avg Win                : ${avg_win:>14,.2f}')
    print(f'    Avg Loss               : ${avg_loss:>14,.2f}')
    print(f'    Largest Win            : ${largest_win:>14,.2f}')
    print(f'    Largest Loss           : ${largest_loss:>14,.2f}')
    print(f'    Profit Factor          : {profit_factor:>13.2f}')
    print(f'    Payoff Ratio           : {payoff_ratio:>13.2f}')
    print(f'    Expectancy / Trade     : ${expectancy:>14,.2f}')
    print(f'    Kelly Criterion        : {kelly:>13.2f}')

    print('\n  ── CONSISTENCY ──')
    print(f'    Max Consec. Wins       : {max_consec_wins:>13}')
    print(f'    Max Consec. Losses     : {max_consec_losses:>13}')
    print(f'    P&L Skewness           : {skewness:>13.2f}')
    print(f'    P&L Kurtosis           : {kurtosis:>13.2f}')

    print('\n  ── EFFICIENCY ──')
    print(f'    Avg Hold Time          : {avg_hold:>12.1f} ticks')
    print(f'    Time in Market         : {time_in_market_pct:>12.1f}%')
    print(f'    Return / Tick in Mkt   : ${return_per_tick:>14,.2f}')
    print(f'    Total Fees Paid        : ${total_fees:>14,.2f}')
    print(f'    Fee Drag (% of gross)  : {fee_drag_pct:>12.1f}%')

    print('\n' + '═' * 62)

    return metrics


# ─────────────────────────────────────────────────
# USAGE: paste this after your backtest runs
# ─────────────────────────────────────────────────
# metrics = compute_all_metrics(trades_df, equity_df, INITIAL_CAPITAL)


In [ ]:
metrics = compute_all_metrics(trades_df, equity_df, INITIAL_CAPITAL)

---
## 37. Final Summary — Complete System

```
  RAW DATA (2,000 ticks × 3 exchanges)
       │
       ▼
  ┌───────────────────────────────────────────┐
  │  FILTER 1: Spread > 5 bps                │  ✅
  │  FILTER 2: Net spread > 0 after costs    │  ✅
  │  FILTER 3: Persists 3+ ticks             │  ✅
  │  FILTER 4: |z-score| > 2                 │  ✅
  │  FILTER 5: Vol-adjusted > 1.5            │  ✅
  └──────────────────┬────────────────────────┘
                     ▼
              TRADEABLE SIGNALS
                     │
                     ▼
  ┌───────────────────────────────────────────┐
  │  BACKTEST ENGINE                          │  ✅
  │  • Group consecutive signals              │
  │  • Enter at signal + 1 tick latency       │
  │  • Exit: z-revert | stop-loss | max hold  │
  │  • Compute realized P&L after fees        │
  └──────────────────┬────────────────────────┘
                     ▼
            📈 PERFORMANCE REPORT
              + Equity Curve
              + Trade Log
              + 8-Panel Dashboard
```
